# M6 W6 v2 full retrieval comparison — four-book Gold v0.1.3

This GitHub-ready notebook evaluates nine pinned embedding models, a BM25
baseline, pure dense retrieval, and six weighted-RRF hybrid settings against the
four-book M4 corpus and `gold_v0_1_3.jsonl`.

The corpus contains 6,184 chunks: 5,976 unchanged OpenStax chunks plus 208 CK-12
chunks. Gold v0.1.3 contains 341 answerable questions and 352 Golden Evidence
spans. The notebook deterministically derives 273 Dev and 68 stratified Test
questions while keeping questions that share a rebound evidence chunk in the
same split.

The lightweight, reviewable result tables and run manifest are committed under
`docs/results/m6_w6_v2_four_book_gold_v0_1_3/`. Large source/index ZIPs and local
FAISS artifacts remain outside Git; see `eval_inputs/m6_v2/README.md` for exact
file names, SHA-256 values, and environment variables.

By default the notebook performs provenance, evidence-binding, split, and saved
result checks without running the embedding models. Set
`CS30_M6_RUN_MODELS=1` for the expensive full comparison. This remains a
development/diagnostic experiment because the supplied corpus is marked
`reportable=false`, Gold annotations are `draft`, and the 273/68 split is not an
officially frozen release split.


## Results overview - 273 Dev / 68 Test rerun

The full notebook completed successfully on 2026-10-22. It evaluated all 64
retrieval configurations on the deterministic 273-question Dev split and the
68-question stratified Test split. Exact Golden Evidence overlap is zero.

Using the predeclared selection rule (complete Hit@5, then MRR), Dev selected
**BGE-M3 with Dense 0.10 / BM25 0.90**. On Dev it achieved complete Hit@5
79.1%, MRR 0.5703, and Hit@10 85.3%. On the frozen 68-question Test it achieved
MRR 0.4715, Hit@1 27.9%, Hit@3 64.7%, Hit@5 70.6%, and Hit@10 80.9%.

For audit only, the exploratory all-model Test table has its highest MRR for
**Qwen3-Embedding-0.6B with Dense 0.05 / BM25 0.95** (MRR 0.5019). This
post-hoc comparison must not replace the Dev-frozen configuration result.


In [1]:
from __future__ import annotations
import collections, hashlib, json, math, os, re, subprocess, time, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "cs30").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from the cs30-personalised-rag checkout.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
REPO_ROOT = PROJECT_ROOT
INPUT_DIR = Path(os.getenv("CS30_M6_INPUT_DIR", PROJECT_ROOT / "eval_inputs" / "m6_v2")).expanduser()

def resolve_input(env_name: str, filename: str) -> Path:
    configured = os.getenv(env_name)
    candidate = Path(configured).expanduser() if configured else INPUT_DIR / filename
    if not candidate.is_file():
        raise FileNotFoundError(
            f"Missing {filename}. Set {env_name} or place it under {INPUT_DIR}. "
            "See eval_inputs/m6_v2/README.md."
        )
    return candidate.resolve()

CORPUS_ZIP = resolve_input("CS30_M6_CORPUS_ZIP", "M4_to_M5_four_book_corpus_a1fe99e.zip")
M5_ZIP = resolve_input("CS30_M6_INDEX_ZIP", "M5_to_M6_four_book_2026-10-21.zip")
INTEGRATED_GOLD = resolve_input("CS30_M6_GOLD", "gold_v0_1_3.jsonl")
GITHUB_GOLD = REPO_ROOT / "m3_gold" / "gold_v0_1_1.jsonl"
WORK_DIR = Path(os.getenv(
    "CS30_M6_WORK_DIR", PROJECT_ROOT / "artifacts" / "m6_w6_retrieval_four_book"
)).expanduser()
OUTPUT_DIR = Path(os.getenv(
    "CS30_M6_OUTPUT_DIR", WORK_DIR / "outputs_four_book_gold_v0_1_3_split_273_68"
)).expanduser()
PUBLISHED_RESULT_DIR = PROJECT_ROOT / "docs" / "results" / "m6_w6_v2_four_book_gold_v0_1_3"
INDEX_DIR = WORK_DIR / "indexes_four_book"
MODEL_CACHE = Path(os.getenv("HF_HOME", Path.home() / ".cache" / "huggingface")).expanduser()

RUN_EXPENSIVE = os.getenv("CS30_M6_RUN_MODELS", "0") == "1"
FORMAL_MODE = os.getenv("CS30_M6_FORMAL_MODE", "0") == "1"
GOLD_ACCEPTANCE_POLICY = "user_accepted_for_m6_evaluation"
EXPECTED_CORPUS_ZIP_SHA256 = "sha256:074ff89686fe3649089329f7858505f7a68d228d0284a83394a0c0b5a547a592"
EXPECTED_M5_ZIP_SHA256 = "sha256:2851abf71164e131b9064de0e87385b3b7f38943cbfcbf420dad5a40c0bf8c3e"
EXPECTED_GOLD_SHA256 = "sha256:7d69af416591b64198bf25806424347366eb2123fdcdcf36ad757d37c4f942b8"
EXPECTED_GOLD_SOURCE_CORPUS_VERSIONS = frozenset({
    "2.0.0-dev.m4-structure-v2-main-3553825",
    "ck12-people-physics-2016-09-09",
})
EXPECTED_GOLD_ANNOTATION_VERSION = "m3_gold_v0.1.3"
EXPECTED_GITHUB_GOLD_SHA256 = "sha256:52d8bab9655ae2f4df2a8bf883a7cbc87acbe83dbb11af983bc228c1412839ad"
FROZEN_EVALUATION_SET_VERSION = None
FROZEN_GOLD_ANNOTATION_VERSION = None
DEVICE = os.getenv("CS30_M6_DEVICE", "cuda")
TOP_K = (1, 3, 5, 10)
SEARCH_DEPTH = 50
RETRIEVAL_DEPTH = max(TOP_K)
RRF_K = 60
HYBRID_WEIGHTS = ((0.95, 0.05), (0.90, 0.10), (0.80, 0.20),
                  (0.75, 0.25), (0.50, 0.50), (0.25, 0.75))  # (BM25, dense)
for path in (REPO_ROOT, CORPUS_ZIP, M5_ZIP, INTEGRATED_GOLD, GITHUB_GOLD):
    assert path.exists(), f"Missing input: {path}"
repo_commit = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
repo_worktree_dirty = bool(subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "status", "--porcelain"], text=True
).strip())
print("GitHub commit:", repo_commit)
print("GitHub worktree dirty:", repo_worktree_dirty)
print("Input source: environment overrides or eval_inputs/m6_v2")
print("Published results:", PUBLISHED_RESULT_DIR.relative_to(PROJECT_ROOT))
print("Expensive run enabled:", RUN_EXPENSIVE, "device:", DEVICE)


GitHub commit: d6abcaeb21ae5a7825b893d51eceb0ed19c760cb
GitHub worktree dirty: True
Input source: environment overrides or eval_inputs/m6_v2
Published results: docs\results\m6_w6_v2_four_book_gold_v0_1_3
Expensive run enabled: False device: cuda


## 1. Model registry and input provenance

In [2]:
MODEL_SPECS = {
    "minilm": dict(model="sentence-transformers/all-MiniLM-L6-v2", dimension=384, batch=64),
    "mpnet": dict(model="sentence-transformers/all-mpnet-base-v2", dimension=768, batch=32),
    "e5_base": dict(model="intfloat/e5-base-v2", dimension=768, batch=32, query_prefix="query: ", document_prefix="passage: "),
    "bge_base": dict(model="BAAI/bge-base-en-v1.5", dimension=768, batch=32, query_prefix="Represent this sentence for searching relevant passages: "),
    "jina": dict(model="jinaai/jina-embeddings-v2-small-en", dimension=512, batch=16, trust_remote_code=True),
    "gte_modernbert": dict(model="Alibaba-NLP/gte-modernbert-base", dimension=768, batch=4, trust_remote_code=True, m5_path="gte-modernbert"),
    "nomic": dict(model="nomic-ai/nomic-embed-text-v1.5", dimension=768, batch=8, trust_remote_code=True, query_prefix="search_query: ", document_prefix="search_document: "),
    "bge_m3": dict(model="BAAI/bge-m3", dimension=1024, batch=4, m5_path="bge-m3"),
    "qwen3": dict(model="Qwen/Qwen3-Embedding-0.6B", dimension=1024, batch=2, m5_path="qwen3-embedding"),
}
for spec in MODEL_SPECS.values():
    spec.setdefault("query_prefix", "")
    spec.setdefault("document_prefix", "")
    spec.setdefault("trust_remote_code", False)
assert len(MODEL_SPECS) == 9 and len({s["model"] for s in MODEL_SPECS.values()}) == 9

# The model roster is explicit and revision-pinned below; no external workbook is required.
EXPECTED_MODEL_IDS = {
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "intfloat/e5-base-v2",
    "BAAI/bge-base-en-v1.5",
    "jinaai/jina-embeddings-v2-small-en",
    "Alibaba-NLP/gte-modernbert-base",
    "nomic-ai/nomic-embed-text-v1.5",
    "BAAI/bge-m3",
    "Qwen/Qwen3-Embedding-0.6B",
}
assert {spec["model"] for spec in MODEL_SPECS.values()} == EXPECTED_MODEL_IDS
print("Registered dense models:", len(MODEL_SPECS))

def sha256(data: bytes) -> str:
    return "sha256:" + hashlib.sha256(data).hexdigest()

def read_jsonl_bytes(data: bytes) -> list[dict]:
    return [json.loads(line) for line in data.decode("utf-8-sig").splitlines() if line.strip()]

with zipfile.ZipFile(CORPUS_ZIP) as z:
    corpus_prefixes = {name.removesuffix("records.jsonl") for name in z.namelist() if name.endswith("records.jsonl")}
    assert len(corpus_prefixes) == 1, sorted(corpus_prefixes)
    CORPUS_PREFIX = next(iter(corpus_prefixes))
    corpus_bytes = z.read(CORPUS_PREFIX + "records.jsonl")
    manifest_bytes = z.read(CORPUS_PREFIX + "manifest.json")
    corpus_manifest = json.loads(manifest_bytes)
    chunks = read_jsonl_bytes(corpus_bytes)
    delivered_checksums = z.read(CORPUS_PREFIX + "SHA256SUMS").decode("utf-8")
    for line in delivered_checksums.splitlines():
        expected, filename = line.split(None, 1)
        assert hashlib.sha256(z.read(CORPUS_PREFIX + filename.strip())).hexdigest() == expected, filename

assert sha256(CORPUS_ZIP.read_bytes()) == EXPECTED_CORPUS_ZIP_SHA256
assert sha256(M5_ZIP.read_bytes()) == EXPECTED_M5_ZIP_SHA256
assert sha256(INTEGRATED_GOLD.read_bytes()) == EXPECTED_GOLD_SHA256
assert sha256(corpus_bytes) == corpus_manifest["corpus_hash"]
assert len(chunks) == corpus_manifest["record_count"] == 6184
assert len({c["chunk_id"] for c in chunks}) == len(chunks)
assert {c["textbook_id"] for c in chunks} == set(corpus_manifest["included_textbook_ids"])
assert set(corpus_manifest["included_textbook_ids"]) == set(corpus_manifest["required_textbook_ids"])
assert "ck12_peoples_physics_concepts" in corpus_manifest["included_textbook_ids"]
assert sum(c["textbook_id"] == "ck12_peoples_physics_concepts" for c in chunks) == 208
assert corpus_manifest["reportable"] is False and corpus_manifest["mode"] == "development"

gold_bytes = INTEGRATED_GOLD.read_bytes()
gold = read_jsonl_bytes(gold_bytes)
github_gold_bytes = GITHUB_GOLD.read_bytes()
github_gold = read_jsonl_bytes(github_gold_bytes)
assert sha256(github_gold_bytes) == EXPECTED_GITHUB_GOLD_SHA256
assert len(gold) == 341 and len({g["question_id"] for g in gold}) == len(gold)
assert len(github_gold) == 20
gold_source_corpus_versions = {g["corpus_version"] for g in gold}
assert gold_source_corpus_versions == EXPECTED_GOLD_SOURCE_CORPUS_VERSIONS
assert corpus_manifest["corpus_version"] not in gold_source_corpus_versions
assert GOLD_ACCEPTANCE_POLICY == "user_accepted_for_m6_evaluation"
assert {g["gold_annotation_version"] for g in gold} == {EXPECTED_GOLD_ANNOTATION_VERSION}
assert {g["annotation_status"] for g in gold} == {"draft"}
assert all(g["answerable"] is True for g in gold)
print({"corpus_version": corpus_manifest["corpus_version"], "chunks": len(chunks),
       "textbooks": corpus_manifest["included_textbook_ids"],
       "integrated_gold_rows": len(gold), "integrated_gold_sha256": sha256(gold_bytes),
       "github_gold_rows": len(github_gold), "github_gold_sha256": sha256(github_gold_bytes),
       "gold_acceptance_policy": GOLD_ACCEPTANCE_POLICY,
       "gold_source_corpus_versions": sorted(gold_source_corpus_versions),
       "gold_rebound_to_corpus_version": corpus_manifest["corpus_version"],
       "source_annotation_statuses": dict(collections.Counter(g["annotation_status"] for g in gold)),
       "gold_splits": dict(collections.Counter(g["split"] for g in gold))})

Registered dense models: 9


{'corpus_version': '2.0.0-dev.m4-four-book-main-a1fe99e', 'chunks': 6184, 'textbooks': ['openstax_college_physics_2e', 'openstax_physics', 'openstax_college_physics_ap_2e', 'ck12_peoples_physics_concepts'], 'integrated_gold_rows': 341, 'integrated_gold_sha256': 'sha256:7d69af416591b64198bf25806424347366eb2123fdcdcf36ad757d37c4f942b8', 'github_gold_rows': 20, 'github_gold_sha256': 'sha256:52d8bab9655ae2f4df2a8bf883a7cbc87acbe83dbb11af983bc228c1412839ad', 'gold_acceptance_policy': 'user_accepted_for_m6_evaluation', 'gold_source_corpus_versions': ['2.0.0-dev.m4-structure-v2-main-3553825', 'ck12-people-physics-2016-09-09'], 'gold_rebound_to_corpus_version': '2.0.0-dev.m4-four-book-main-a1fe99e', 'source_annotation_statuses': {'draft': 341}, 'gold_splits': {'proposed_test': 30, 'pending': 311}}


## 2. Bind evidence by source `block_id`

M3's chapter-local offsets and M4's document-wide offsets are different coordinate systems. A Gold span is accepted only when its block ID resolves uniquely, its chapter matches, and its normalized verbatim text occurs in that chunk. Fail closed if any span cannot be bound.

In [3]:
def normalized(text: str) -> str:
    return " ".join(text.casefold().split())

block_to_chunks = collections.defaultdict(set)
by_chunk = {c["chunk_id"]: c for c in chunks}
for chunk in chunks:
    for span in chunk["spans"]:
        block_to_chunks[span["block_id"]].add(chunk["chunk_id"])

binding_rows = []
gold_paths = {}
for question in gold:
    qid = question["question_id"]
    paths = []
    for group in question["gold_core_evidence_sets"]:
        mapped = []
        for span in group:
            candidates = block_to_chunks.get(span.get("block_id"), set())
            assert len(candidates) == 1, (qid, span.get("span_id"), "non-unique/missing block")
            chunk_id = next(iter(candidates))
            chunk = by_chunk[chunk_id]
            assert str(chunk["chapter_id"]) == str(span["chapter_id"]), (qid, "chapter mismatch")
            assert normalized(span["verbatim_text"]) in normalized(chunk["text"]), (qid, "verbatim mismatch")
            mapped.append(chunk_id)
            binding_rows.append({"question_id": qid, "span_id": span["span_id"],
                                 "block_id": span["block_id"], "chunk_id": chunk_id,
                                 "textbook_id": chunk["textbook_id"], "method": "unique_block_id+chapter+verbatim"})
        paths.append(frozenset(mapped))
    assert paths, (qid, "no core evidence")
    gold_paths[qid] = paths
assert len(binding_rows) == 352
assert len(gold_paths) == 341
binding_bytes = ("\n".join(json.dumps(r, sort_keys=True, ensure_ascii=False) for r in binding_rows) + "\n").encode("utf-8")
print("Bound spans:", len(binding_rows), "of", sum(len(p) for q in gold for p in q["gold_core_evidence_sets"]))
print("Binding SHA-256:", sha256(binding_bytes))

Bound spans: 352 of 352
Binding SHA-256: sha256:99b72302fd335676f5ff8b53516972db361e81eca97aa52b0ee050086b4c055e


## 3. Verify the three M5 candidate index artifacts

In [4]:
M5_ARTIFACTS = {}
M5_INDEX_BYTES = {}
ordered_chunk_ids = [c["chunk_id"] for c in chunks]
with zipfile.ZipFile(M5_ZIP) as z:
    for key, spec in MODEL_SPECS.items():
        if "m5_path" not in spec:
            continue
        prefix = f"{spec['m5_path']}/"
        artifact = json.loads(z.read(prefix + "artifact.json"))
        assert z.read(prefix + "records.jsonl") == corpus_bytes
        assert json.loads(z.read(prefix + "manifest.json")) == corpus_manifest
        ids_bytes = z.read(prefix + "index/chunk_ids.json")
        idx_bytes = z.read(prefix + "index/index.faiss")
        for name in ("corpus_version", "corpus_hash", "manifest_hash", "chunk_config_hash"):
            assert artifact[name] == corpus_manifest[name], (key, name)
        assert artifact["required_textbook_ids"] == corpus_manifest["required_textbook_ids"]
        assert artifact["included_textbook_ids"] == corpus_manifest["included_textbook_ids"]
        assert artifact["chunk_count"] == len(chunks)
        assert artifact["chunk_ids"] == ordered_chunk_ids
        assert json.loads(ids_bytes) == ordered_chunk_ids
        assert sha256(ids_bytes) == artifact["metadata"]["chunk_ids_sha256"]
        assert sha256(idx_bytes) == artifact["metadata"]["index_sha256"]
        assert artifact["embedding_model"] == spec["model"]
        assert artifact["embedding_dimension"] == spec["dimension"]
        assert artifact["similarity_metric"] == "inner_product"
        assert artifact["normalise_embeddings"] is True
        assert artifact["index_type"] == "faiss.IndexFlatIP"
        assert not spec["query_prefix"] and not spec["document_prefix"]  # M5 encoder uses plain encode.
        M5_ARTIFACTS[key] = artifact
        M5_INDEX_BYTES[key] = idx_bytes
print(pd.DataFrame([{"model": a["embedding_model"], "revision": a["embedding_revision"],
                     "dimension": a["embedding_dimension"], "chunks": a["chunk_count"]}
                    for a in M5_ARTIFACTS.values()]).to_string(index=False))

                          model                                 revision  dimension  chunks
Alibaba-NLP/gte-modernbert-base e7f32e3c00f91d699e8c43b53106206bcc72bb22        768    6184
                    BAAI/bge-m3 5617a9f61b028005a4858fdac845db406aefb181       1024    6184
      Qwen/Qwen3-Embedding-0.6B 97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3       1024    6184


## 4. Strict formal gate and diagnostic split

In [5]:
EVALUATION_SET_VERSION = "m6-v2-working-gold-341-v0.1.3-split-273-68-2026-10-22"
GOLD_ANNOTATION_VERSION = sorted({g["gold_annotation_version"] for g in gold})

def assert_formal_ready() -> None:
    assert corpus_manifest["reportable"] is True, "M4 corpus is development-only."
    assert corpus_manifest["mode"] != "development"
    assert "ck12" in " ".join(corpus_manifest["included_textbook_ids"]).lower(), "CK-12 source absent."
    assert GOLD_ACCEPTANCE_POLICY == "user_accepted_for_m6_evaluation"
    assert {g["split"] for g in gold} <= {"dev", "test"}, "Dev/Test splits are not frozen."
    assert all(g["corpus_version"] == corpus_manifest["corpus_version"] for g in gold), "Formal Gold must be native to the frozen corpus."
    assert FROZEN_EVALUATION_SET_VERSION, "Missing approved evaluation_set_version."
    assert FROZEN_GOLD_ANNOTATION_VERSION, "Missing frozen Gold annotation version."
    assert GOLD_ANNOTATION_VERSION == [FROZEN_GOLD_ANNOTATION_VERSION]

if FORMAL_MODE:
    assert_formal_ready()
else:
    print("Gold accepted for M6 scoring after verified evidence rebinding; the derived 273/68 split remains diagnostic-only.")
    try:
        assert_formal_ready()
    except AssertionError as exc:
        print("Formal gate correctly rejects these inputs:", exc)
    else:
        raise AssertionError("Formal gate unexpectedly accepted development inputs")

# Deterministic group-stratified 80/20 split. Questions sharing any rebound
# evidence chunk form one indivisible group so exact Golden Evidence cannot leak
# between Dev and Test.
TEST_QUOTAS = {
    "openstax_college_physics_2e": 25,
    "openstax_college_physics_ap_2e": 19,
    "ck12_peoples_physics_concepts": 17,
    "openstax_physics": 7,
}
SPLIT_SEED = "m6-v2-gold-v0.1.3-stratified-80-20"
STRATIFY_FIELDS = ("question_type", "question_difficulty", "source_split")

def question_textbook_id(question: dict) -> str:
    textbook_ids = {
        by_chunk[chunk_id]["textbook_id"]
        for path in gold_paths[question["question_id"]]
        for chunk_id in path
    }
    assert len(textbook_ids) == 1, (question["question_id"], textbook_ids)
    return next(iter(textbook_ids))

question_by_id = {g["question_id"]: g for g in gold}
parent = {question_id: question_id for question_id in question_by_id}

def find(question_id: str) -> str:
    while parent[question_id] != question_id:
        parent[question_id] = parent[parent[question_id]]
        question_id = parent[question_id]
    return question_id

def union(left: str, right: str) -> None:
    left_root, right_root = find(left), find(right)
    if left_root != right_root:
        parent[right_root] = left_root

questions_by_evidence_chunk = collections.defaultdict(list)
for question in gold:
    evidence_chunks = set().union(*gold_paths[question["question_id"]])
    for chunk_id in evidence_chunks:
        questions_by_evidence_chunk[chunk_id].append(question["question_id"])
for question_ids in questions_by_evidence_chunk.values():
    for other_id in question_ids[1:]:
        union(question_ids[0], other_id)

component_rows = collections.defaultdict(list)
for question in gold:
    component_rows[find(question["question_id"])].append(question)
components = list(component_rows.values())
assert sum(map(len, components)) == len(gold)
assert all(len({question_textbook_id(q) for q in group}) == 1 for group in components)

def component_hash(group: list[dict]) -> str:
    question_ids = "|".join(sorted(q["question_id"] for q in group))
    return hashlib.sha256(f"{SPLIT_SEED}|{question_ids}".encode()).hexdigest()

def select_test_components(textbook_id: str, quota: int) -> list[list[dict]]:
    candidates = [group for group in components if question_textbook_id(group[0]) == textbook_id]
    population = [question for group in candidates for question in group]
    population_counts = {
        field: collections.Counter(question[field] for question in population)
        for field in STRATIFY_FIELDS
    }
    selected, selected_rows = [], []
    remaining = list(candidates)
    while len(selected_rows) < quota:
        room = quota - len(selected_rows)
        feasible = [group for group in remaining if len(group) <= room]
        assert feasible, (textbook_id, room)

        def imbalance(group: list[dict]) -> tuple[float, str]:
            proposed = selected_rows + group
            penalty = 0.0
            for field in STRATIFY_FIELDS:
                observed = collections.Counter(question[field] for question in proposed)
                for value, population_count in population_counts[field].items():
                    target = len(proposed) * population_count / len(population)
                    penalty += (observed[value] - target) ** 2 / max(target, 1.0)
            return penalty, component_hash(group)

        chosen = min(feasible, key=imbalance)
        selected.append(chosen)
        selected_rows.extend(chosen)
        remaining.remove(chosen)
    assert len(selected_rows) == quota
    return selected

test_components = []
for textbook_id, quota in TEST_QUOTAS.items():
    test_components.extend(select_test_components(textbook_id, quota))
test_question_ids = {q["question_id"] for group in test_components for q in group}
assert len(test_question_ids) == sum(TEST_QUOTAS.values()) == 68
TEST_QUESTION_IDS_SHA256 = sha256("\n".join(sorted(test_question_ids)).encode())

dev_gold = [
    {**g, "source_gold_split": g["split"], "evaluation_split": "dev"}
    for g in gold if g["question_id"] not in test_question_ids
]
heldout_gold = [
    {**g, "source_gold_split": g["split"], "evaluation_split": "stratified_test"}
    for g in gold if g["question_id"] in test_question_ids
]
assert len(dev_gold) == 273 and len(heldout_gold) == 68
assert not ({g["question_id"] for g in dev_gold} & {g["question_id"] for g in heldout_gold})
assert collections.Counter(question_textbook_id(g) for g in heldout_gold) == collections.Counter(TEST_QUOTAS)
assert {g["question_type"] for g in heldout_gold} == {"definition", "identification", "causal", "application", "comparison", "calculation"}
dev_evidence = {chunk_id for q in dev_gold for path in gold_paths[q["question_id"]] for chunk_id in path}
test_evidence = {chunk_id for q in heldout_gold for path in gold_paths[q["question_id"]] for chunk_id in path}
assert not (dev_evidence & test_evidence), "Golden Evidence leakage across the derived split."
print("Diagnostic development:", len(dev_gold), "stratified Test:", len(heldout_gold))
print("Test textbooks:", dict(collections.Counter(question_textbook_id(g) for g in heldout_gold)))
print("Test question types:", dict(collections.Counter(g["question_type"] for g in heldout_gold)))
print("Test difficulties:", dict(collections.Counter(g["question_difficulty"] for g in heldout_gold)))
print("Shared evidence chunks across splits:", len(dev_evidence & test_evidence))


Gold accepted for M6 scoring after verified evidence rebinding; the derived 273/68 split remains diagnostic-only.
Formal gate correctly rejects these inputs: M4 corpus is development-only.
Diagnostic development: 273 stratified Test: 68
Test textbooks: {'openstax_college_physics_2e': 25, 'openstax_college_physics_ap_2e': 19, 'openstax_physics': 7, 'ck12_peoples_physics_concepts': 17}
Test question types: {'identification': 21, 'causal': 17, 'definition': 22, 'comparison': 2, 'application': 5, 'calculation': 1}
Test difficulties: {'easy': 41, 'medium': 24, 'hard': 3}
Shared evidence chunks across splits: 0


## 5. BM25, dense and weighted RRF (m6local retrieval method)

In [6]:
STOPWORDS = frozenset({
    "a", "an", "and", "are", "as", "at", "be", "by", "can", "do", "does",
    "for", "from", "how", "in", "is", "it", "of", "on", "or", "that", "the",
    "to", "was", "were", "what", "when", "where", "which", "who", "why",
    "will", "with",
})
REMOVE_BM25_STOPWORDS = True
BM25_K1 = 1.5
BM25_B = 0.75
TOKEN_RE = re.compile(r"[a-z0-9]+(?:['-][a-z0-9]+)*", re.IGNORECASE)
def tokenize(text: str, remove_stopwords: bool = REMOVE_BM25_STOPWORDS) -> list[str]:
    tokens = TOKEN_RE.findall(text.casefold())
    return [token for token in tokens if token not in STOPWORDS] if remove_stopwords else tokens

doc_tokens = [tokenize(c["text"]) for c in chunks]
doc_lengths = np.asarray([len(t) for t in doc_tokens], dtype=np.float64)
avg_doc_length = float(doc_lengths.mean())
postings = collections.defaultdict(list)
for i, tokens in enumerate(doc_tokens):
    for term, freq in collections.Counter(tokens).items():
        postings[term].append((i, freq))

def bm25_search(query: str, depth: int = SEARCH_DEPTH) -> list[tuple[str, float]]:
    scores = collections.defaultdict(float)
    for term, query_freq in collections.Counter(tokenize(query)).items():
        rows = postings.get(term, ())
        df = len(rows)
        if not df:
            continue
        idf = math.log(1.0 + (len(chunks) - df + 0.5) / (df + 0.5))
        for row, freq in rows:
            denom = freq + BM25_K1 * (1.0 - BM25_B + BM25_B * doc_lengths[row] / avg_doc_length)
            score = query_freq * idf * freq * (BM25_K1 + 1.0) / denom
            scores[row] += score
    ranked = sorted(scores, key=lambda i: (-scores[i], ordered_chunk_ids[i]))[:depth]
    return [(ordered_chunk_ids[i], float(scores[i])) for i in ranked]

def weighted_rrf(bm25_hits, dense_hits, bm25_weight: float, dense_weight: float,
                 depth: int = RETRIEVAL_DEPTH) -> list[tuple[str, float]]:
    assert math.isclose(bm25_weight + dense_weight, 1.0, abs_tol=1e-9)
    fused = collections.defaultdict(float)
    best_rank = {}
    for weight, hits in ((dense_weight, dense_hits), (bm25_weight, bm25_hits)):
        for rank, (chunk_id, _) in enumerate(hits[:SEARCH_DEPTH], 1):
            fused[chunk_id] += weight / (RRF_K + rank)
            best_rank[chunk_id] = min(rank, best_rank.get(chunk_id, math.inf))
    return sorted(fused.items(), key=lambda row: (-row[1], best_rank[row[0]], row[0]))[:depth]

assert tokenize("The force and force") == ["force", "force"]
assert tokenize("the and") == []
METHOD_CONFIG = {
    "bm25_stopwords_enabled": REMOVE_BM25_STOPWORDS,
    "bm25_stopwords": sorted(STOPWORDS),
    "bm25_k1": BM25_K1,
    "bm25_b": BM25_B,
    "bm25_query_term_frequency": True,
    "rrf_k": RRF_K,
    "rrf_tie_break": "minimum_candidate_rank_then_chunk_id",
    "candidate_depth": SEARCH_DEPTH,
    "final_depth": RETRIEVAL_DEPTH,
}
print("BM25 stopwords:", REMOVE_BM25_STOPWORDS, "k1:", BM25_K1, "b:", BM25_B)

def load_or_build_dense(model_key: str):
    # Return (SentenceTransformer, FAISS index, artifact); write only compatible local indexes.
    import faiss
    from sentence_transformers import SentenceTransformer
    spec = MODEL_SPECS[model_key]
    revision = M5_ARTIFACTS[model_key]["embedding_revision"] if model_key in M5_ARTIFACTS else None
    model = SentenceTransformer(spec["model"], revision=revision, device=DEVICE,
                                trust_remote_code=spec["trust_remote_code"], cache_folder=str(MODEL_CACHE))
    if model_key in M5_ARTIFACTS:
        artifact = M5_ARTIFACTS[model_key]
        index = faiss.deserialize_index(np.frombuffer(M5_INDEX_BYTES[model_key], dtype="uint8"))
    else:
        location = INDEX_DIR / model_key
        artifact_path = location / "artifact.json"
        index_path = location / "index.faiss"
        policy = {"corpus_hash": corpus_manifest["corpus_hash"], "model": spec["model"],
                  "query_prefix": spec["query_prefix"], "document_prefix": spec["document_prefix"],
                  "metric": "inner_product", "normalise_embeddings": True,
                  "chunk_ids_sha256": sha256(json.dumps(ordered_chunk_ids, separators=(',', ':')).encode())}
        if artifact_path.is_file() and index_path.is_file():
            artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
            assert all(artifact.get(k) == v for k, v in policy.items()), f"Stale local index: {location}"
            assert sha256(index_path.read_bytes()) == artifact["index_sha256"]
            index = faiss.read_index(str(index_path))
        else:
            texts = [spec["document_prefix"] + (c.get("embed_text") or c["text"]) for c in chunks]
            vectors = model.encode(texts, batch_size=spec["batch"], normalize_embeddings=True,
                                   convert_to_numpy=True, show_progress_bar=True).astype("float32")
            assert vectors.shape == (len(chunks), spec["dimension"])
            assert np.isfinite(vectors).all()
            faiss.normalize_L2(vectors)
            index = faiss.IndexFlatIP(spec["dimension"])
            index.add(vectors)
            location.mkdir(parents=True, exist_ok=True)
            faiss.write_index(index, str(index_path))
            resolved = getattr(getattr(getattr(model[0], "auto_model", None), "config", None), "_commit_hash", None)
            artifact = {**policy, "embedding_revision": str(resolved or "unresolved"),
                        "dimension": spec["dimension"], "chunk_count": len(chunks),
                        "index_sha256": sha256(index_path.read_bytes()), "built_at_utc": datetime.now(timezone.utc).isoformat()}
            artifact_path.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
    assert index.ntotal == len(chunks) and index.d == spec["dimension"]
    assert isinstance(index, faiss.IndexFlatIP), "Only normalized FlatIP is supported."
    return model, index, artifact

def dense_search(query: str, model_key: str, model, index, depth: int = SEARCH_DEPTH):
    import faiss
    spec = MODEL_SPECS[model_key]
    query_vec = model.encode([spec["query_prefix"] + query], normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=False).astype("float32")
    assert query_vec.shape == (1, spec["dimension"]) and np.isfinite(query_vec).all()
    faiss.normalize_L2(query_vec)  # Matches artifact normalise_embeddings=True and FlatIP.
    scores, positions = index.search(query_vec, min(depth, len(chunks)))
    return [(ordered_chunk_ids[int(i)], float(score)) for score, i in zip(scores[0], positions[0]) if i >= 0]

BM25 stopwords: True k1: 1.5 b: 0.75


## 6. Evidence retrieval metrics and grouping

In [7]:
def score_one(question: dict, hits: list[tuple[str, float]], model_key: str,
              mode: str, bm25_weight=None, dense_weight=None) -> dict:
    ids = [chunk_id for chunk_id, _ in hits]
    assert len(ids) == len(set(ids)), (question["question_id"], model_key, mode, "duplicate hits")
    qid = question["question_id"]
    answerable = question.get("answerable", True) is True
    paths = gold_paths[qid] if answerable else []
    relevant = set().union(*paths) if paths else set()
    rank = {chunk_id: i for i, chunk_id in enumerate(ids, 1)}
    first_rank = min((rank[cid] for cid in relevant if cid in rank), default=None)
    complete_rank = min((max((rank.get(cid, math.inf) for cid in path), default=math.inf)
                         for path in paths), default=math.inf)
    complete_rank = None if math.isinf(complete_rank) else int(complete_rank)
    source_ids = {by_chunk[cid]["textbook_id"] for cid in relevant}
    providers = {by_chunk[cid]["provider"] for cid in relevant}
    row = {"question_id": qid, "source_split": question["source_split"],
           "evaluation_split": question.get("evaluation_split", question["split"]),
           "source_gold_split": question.get("source_gold_split", question["split"]),
           "annotation_status": question["annotation_status"],
           "model": model_key, "mode": mode, "bm25_weight": bm25_weight,
           "dense_weight": dense_weight, "answerable": answerable,
           "scope": ("unanswerable" if not answerable else "cross_textbook" if len(source_ids) > 1 else "single_source"),
           "provider_scope": ("unanswerable" if not answerable else "cross_provider" if len(providers) > 1 else "single_provider"),
           "gold_textbook": "+".join(sorted(source_ids)), "gold_provider": "+".join(sorted(providers)),
           "first_relevant_rank": first_rank, "complete_evidence_rank": complete_rank,
           "mrr": 0.0 if first_rank is None else 1 / first_rank,
           "complete_evidence_mrr": 0.0 if complete_rank is None else 1 / complete_rank,
           "returned_count": len(hits), "empty_retrieval": int(not hits),
           "unanswerable_no_hit": int(not hits) if not answerable else None,
           "unanswerable_false_positive": int(bool(hits)) if not answerable else None,
           "ranked_chunk_ids": ids}
    for k in TOP_K:
        top = set(ids[:k])
        row[f"hit_at_{k}"] = int(bool(top & relevant))
        row[f"complete_hit_at_{k}"] = int(any(path <= top for path in paths))
        row[f"evidence_recall_at_{k}"] = max((len(path & top) / len(path) for path in paths), default=0.0) if answerable else None
        row[f"core_precision_at_{k}"] = len(top & relevant) / len(top) if top and answerable else None
        row[f"no_relevant_hit_at_{k}"] = int(not (top & relevant)) if answerable else None
    return row

def summarize(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    # The integrated Gold has no partial-evidence spans. Thus noise is exactly
    # 1 - core precision for nonempty retrieval, and partial count is zero.
    assert all(not q.get("partial_evidence") for q in gold)
    frame["run_name"] = frame["model"] + "_" + frame["mode"]
    frame["first_hit_mrr"] = frame["mrr"]
    for k in TOP_K:
        frame[f"complete_evidence_hit_at_{k}"] = frame[f"complete_hit_at_{k}"]
        frame[f"recall_at_{k}"] = frame[f"evidence_recall_at_{k}"]
        frame[f"returned_count_at_{k}"] = frame["returned_count"].clip(upper=k)
        frame[f"empty_retrieval_rate_at_{k}"] = frame[f"returned_count_at_{k}"].eq(0).astype(float)
        frame[f"no_hit_rate_at_{k}"] = frame[f"empty_retrieval_rate_at_{k}"]
        frame[f"no_relevant_hit_rate_at_{k}"] = frame[f"no_relevant_hit_at_{k}"].where(
            frame[f"returned_count_at_{k}"].gt(0))
        frame[f"context_noise_rate_at_{k}"] = 1 - frame[f"core_precision_at_{k}"]
        frame[f"partial_evidence_count_at_{k}"] = 0
    metrics = ["mrr", "complete_evidence_mrr"] + [
        f"{metric}_at_{k}" for metric in ("hit", "complete_evidence_hit", "evidence_recall") for k in TOP_K
    ] + ["first_hit_mrr"] + [
        f"{metric}_at_{k}" for metric in ("complete_hit", "recall", "core_precision",
            "context_noise_rate", "no_relevant_hit_rate", "partial_evidence_count",
            "returned_count", "empty_retrieval_rate", "no_hit_rate") for k in TOP_K
    ] + ["returned_count", "empty_retrieval", "unanswerable_no_hit", "unanswerable_false_positive"] + [
        f"no_relevant_hit_at_{k}" for k in TOP_K]
    groups = ["run_name", "model", "mode", "dense_weight", "bm25_weight"]
    grouped = frame.groupby(groups, dropna=False)
    counts = grouped.size().rename("question_count")
    nullable = [f"{metric}_at_{k}" for metric in ("core_precision", "context_noise_rate",
        "no_relevant_hit_rate", "partial_evidence_count") for k in TOP_K]
    defined = grouped[nullable].count().rename(columns={column: f"{column}_defined_count" for column in nullable})
    return counts.to_frame().join(grouped[metrics].mean()).join(defined).reset_index()

def grouped_summary(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    groups = ["model", "mode", "evaluation_split", "scope", "provider_scope", "gold_textbook", "gold_provider"]
    return frame.groupby(groups, dropna=False).agg(question_count=("question_id", "nunique"),
        hit_at_5=("hit_at_5", "mean"), complete_hit_at_5=("complete_hit_at_5", "mean"),
        evidence_recall_at_5=("evidence_recall_at_5", "mean"), mrr=("mrr", "mean")).reset_index()

# Tiny synthetic tests only verify metric logic; they are never merged with real output.
_q = {"question_id": dev_gold[0]["question_id"], "source_split": "train", "split": "pending", "annotation_status": dev_gold[0]["annotation_status"]}
_a = next(iter(gold_paths[_q["question_id"]][0]))
assert score_one(_q, [(_a, 1.0)], "fixture", "fixture")["hit_at_1"] == 1
assert score_one(_q, [], "fixture", "fixture")["empty_retrieval"] == 1
assert score_one(_q, [], "fixture", "fixture")["no_relevant_hit_at_5"] == 1
_u = {**_q, "answerable": False}
assert score_one(_u, [], "fixture", "fixture")["unanswerable_no_hit"] == 1
assert score_one(_u, [(_a, 1.0)], "fixture", "fixture")["unanswerable_false_positive"] == 1
_joint = next(q for q in gold if any(len(path) > 1 for path in gold_paths[q["question_id"]]))
_joint_ids = list(next(path for path in gold_paths[_joint["question_id"]] if len(path) > 1))
assert score_one(_joint, [(_joint_ids[0], 1.0)], "fixture", "fixture")["hit_at_1"] == 1
assert score_one(_joint, [(_joint_ids[0], 1.0)], "fixture", "fixture")["complete_hit_at_1"] == 0
assert score_one(_joint, [(cid, 1.0) for cid in _joint_ids], "fixture", "fixture")["complete_hit_at_3"] == 1
print("Metric smoke tests passed.")

Metric smoke tests passed.


## 7. Run real diagnostic retrieval

`RUN_EXPENSIVE=False` performs all input, mapping and index-provenance checks
above but **does not** evaluate models. When enabled, every model is run on the
311 pending rows for diagnostic comparison. The 30 stratified Test rows are not
used for configuration selection. All current questions are answerable and all
Golden Evidence paths are single-textbook/single-provider; CK-12-specific
performance is reported through the grouped output, while cross-textbook and
cross-provider rates remain unsupported.


In [8]:
def evaluate_questions(questions: list[dict], include_models: list[str], checkpoint_prefix: str = "dev"):
    rows, candidates, index_provenance = [], [], {}
    assert checkpoint_prefix in ("dev", "proposed_test_all"), checkpoint_prefix
    progress_path = OUTPUT_DIR / ("progress.json" if checkpoint_prefix == "dev" else f"{checkpoint_prefix}_progress.json")
    previous = json.loads(progress_path.read_text(encoding="utf-8")) if progress_path.is_file() else {}
    if previous:
        assert previous.get("checkpoint_prefix", checkpoint_prefix) == checkpoint_prefix
        assert previous["corpus_hash"] == corpus_manifest["corpus_hash"]
        assert previous["gold_sha256"] == sha256(gold_bytes)
        assert previous["total_dense_models"] == len(include_models)
        assert previous["hybrid_weights"] == [list(pair) for pair in HYBRID_WEIGHTS]
        assert previous["method_config"] == METHOD_CONFIG
    completed_models = list(previous.get("completed_models", []))
    def recover(key: str):
        if key not in completed_models:
            return None
        metrics_path = OUTPUT_DIR / f"{checkpoint_prefix}_{key}_metrics.jsonl"
        candidates_path = OUTPUT_DIR / f"{checkpoint_prefix}_{key}_candidate_hits.jsonl"
        assert metrics_path.is_file() and candidates_path.is_file(), key
        old_rows = read_jsonl_bytes(metrics_path.read_bytes())
        old_candidates = read_jsonl_bytes(candidates_path.read_bytes())
        expected = len(questions) if key == "bm25" else len(questions) * (1 + len(HYBRID_WEIGHTS))
        assert len(old_rows) == expected and {r["question_id"] for r in old_rows} == {q["question_id"] for q in questions}
        return old_rows, old_candidates
    def checkpoint(key: str, new_rows: list[dict], new_candidates: list[dict]):
        # Keep a per-model recovery point; no result is marked complete before its model finishes.
        (OUTPUT_DIR / f"{checkpoint_prefix}_{key}_metrics.jsonl").write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in new_rows), encoding="utf-8")
        (OUTPUT_DIR / f"{checkpoint_prefix}_{key}_candidate_hits.jsonl").write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in new_candidates), encoding="utf-8")
        if key not in completed_models:
            completed_models.append(key)
        progress_path.write_text(json.dumps({
            "completed_models": completed_models, "total_dense_models": len(include_models),
            "corpus_hash": corpus_manifest["corpus_hash"], "gold_sha256": sha256(gold_bytes),
            "hybrid_weights": [list(pair) for pair in HYBRID_WEIGHTS],
            "method_config": METHOD_CONFIG,
            "checkpoint_prefix": checkpoint_prefix,
            "updated_at_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
    bm25_by_qid = {q["question_id"]: bm25_search(q["question"], SEARCH_DEPTH) for q in questions}
    recovered = recover("bm25")
    if recovered:
        rows.extend(recovered[0]); candidates.extend(recovered[1])
        print("Recovered completed bm25 checkpoint", flush=True)
    else:
        for q in questions:
            hits = bm25_by_qid[q["question_id"]][:RETRIEVAL_DEPTH]
            rows.append(score_one(q, hits, "bm25", "bm25"))
            for rank, (chunk_id, score) in enumerate(hits, 1):
                candidates.append({"question_id": q["question_id"], "model": "bm25", "mode": "bm25",
                                   "rank": rank, "chunk_id": chunk_id, "score": score,
                                   "query": q["question"], "corpus_hash": corpus_manifest["corpus_hash"],
                                   "textbook_id": by_chunk[chunk_id]["textbook_id"],
                                   "provider": by_chunk[chunk_id]["provider"],
                                   "source_locator": by_chunk[chunk_id]["source_locator"]})
        checkpoint("bm25", rows, candidates)
    for key in include_models:
        recovered = recover(key)
        if recovered:
            rows.extend(recovered[0]); candidates.extend(recovered[1])
            index_provenance[key] = (M5_ARTIFACTS[key] if key in M5_ARTIFACTS else
                json.loads((INDEX_DIR / key / "artifact.json").read_text(encoding="utf-8")))
            print("Recovered completed", key, "checkpoint", flush=True)
            continue
        print("Running", key, MODEL_SPECS[key]["model"], flush=True)
        row_start, candidate_start = len(rows), len(candidates)
        model, index, artifact = load_or_build_dense(key)
        index_provenance[key] = artifact
        for q in questions:
            qid = q["question_id"]
            dense = dense_search(q["question"], key, model, index, SEARCH_DEPTH)
            variants = [("dense", None, None, dense[:RETRIEVAL_DEPTH])]
            variants += [(f"hybrid_d{dw:.2f}_b{bw:.2f}", bw, dw, weighted_rrf(bm25_by_qid[qid], dense, bw, dw))
                         for bw, dw in HYBRID_WEIGHTS]
            for mode, bw, dw, hits in variants:
                rows.append(score_one(q, hits, key, mode, bw, dw))
                for rank, (chunk_id, score) in enumerate(hits, 1):
                    candidates.append({"question_id": qid, "model": key, "mode": mode,
                                       "rank": rank, "chunk_id": chunk_id, "score": score,
                                       "query": q["question"], "corpus_hash": corpus_manifest["corpus_hash"],
                                       "textbook_id": by_chunk[chunk_id]["textbook_id"],
                                       "provider": by_chunk[chunk_id]["provider"],
                                       "source_locator": by_chunk[chunk_id]["source_locator"]})
        checkpoint(key, rows[row_start:], candidates[candidate_start:])
        del model, index
        try:
            import torch
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        except ImportError:
            pass
    return rows, candidates, index_provenance

if RUN_EXPENSIVE:
    import faiss, sentence_transformers, torch
    from IPython.display import display
    if DEVICE == "cuda": assert torch.cuda.is_available(), "CUDA requested but unavailable."
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    INDEX_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_CACHE.mkdir(parents=True, exist_ok=True)
    dev_rows, dev_candidates, index_provenance = evaluate_questions(dev_gold, list(MODEL_SPECS))
    dev_summary = summarize(dev_rows).sort_values(["complete_hit_at_5", "mrr"], ascending=False)
    display(dev_summary)
else:
    dev_rows = dev_candidates = []
    index_provenance = {}
    print("Preflight only: model execution skipped; committed result tables are loaded below.")

Preflight only: model execution skipped; committed result tables are loaded below.


## 8. Per-model development results (m6local layout)

Each model has one pure Dense row and six hybrid rows. The weight columns always
mean **Dense / BM25**, in that order: 0.05/0.95, 0.10/0.90, 0.20/0.80,
0.25/0.75, 0.50/0.50, 0.75/0.25. BM25 alone is shown separately. These are
273-question diagnostic-development results, not final Test results.

The wide tables include every available m6local metric and v2-specific
diagnostics, including defined-question counts for nullable metrics. Values are
regenerated from Gold v0.1.3 and saved under the isolated output directory.


In [9]:
from IPython.display import display

def result_path(name: str) -> Path:
    local = OUTPUT_DIR / name
    published = PUBLISHED_RESULT_DIR / name
    if local.is_file():
        return local
    if published.is_file():
        return published
    return local

def saved_result_exists(name: str) -> bool:
    return result_path(name).is_file()

def model_table(key: str, split: str = "dev") -> pd.DataFrame:
    assert split in ("dev", "proposed_test_all"), split
    live_name = "dev_summary" if split == "dev" else "proposed_test_all_summary"
    if live_name not in globals():
        saved = result_path(f"{split}_summary.csv")
        if not saved.is_file():
            raise FileNotFoundError(f"Run the full comparison first: {saved}")
        frame = pd.read_csv(saved)
    else:
        frame = globals()[live_name]
    table = frame.loc[frame["model"].eq(key)].copy()
    if table.empty:
        raise ValueError(f"No measured rows for {key}")
    if "run_name" not in table:
        table["run_name"] = table["model"] + "_" + table["mode"]
    table["sort_weight"] = table["dense_weight"].fillna(-1)
    table = table.sort_values("sort_weight")
    table["mode"] = table["mode"].map(lambda mode: "hybrid" if mode.startswith("hybrid_") else mode)
    leading = ["run_name", "model", "mode", "dense_weight", "bm25_weight", "question_count"]
    columns = leading + [column for column in table.columns if column not in leading + ["sort_weight"]]
    return table[columns].reset_index(drop=True)

### Dev: BM25 baseline (273 questions)

In [10]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("bm25"))
else:
    print("Dev results pending: BM25 baseline")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bm25_bm25,bm25,bm25,NaN,NaN,273,0.57286,0.570418,0.395604,0.747253,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: MiniLM (273 questions)

In [11]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("minilm"))
else:
    print("Dev results pending: MiniLM")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,minilm_dense,minilm,dense,NaN,NaN,273,0.281771,0.277741,0.139194,0.391941,...,273,273,273,273,273,273,273,273,273,273
1,minilm_hybrid_d0.05_b0.95,minilm,hybrid,0.05,0.95,273,0.551779,0.549337,0.369963,0.717949,...,273,273,273,273,273,273,273,273,273,273
2,minilm_hybrid_d0.10_b0.90,minilm,hybrid,0.10,0.90,273,0.534599,0.532157,0.355311,0.684982,...,273,273,273,273,273,273,273,273,273,273
3,minilm_hybrid_d0.20_b0.80,minilm,hybrid,0.20,0.80,273,0.510265,0.507823,0.344322,0.644689,...,273,273,273,273,273,273,273,273,273,273
4,minilm_hybrid_d0.25_b0.75,minilm,hybrid,0.25,0.75,273,0.493651,0.491209,0.329670,0.619048,...,273,273,273,273,273,273,273,273,273,273
5,minilm_hybrid_d0.50_b0.50,minilm,hybrid,0.50,0.50,273,0.436492,0.434050,0.285714,0.534799,...,273,273,273,273,273,273,273,273,273,273
6,minilm_hybrid_d0.75_b0.25,minilm,hybrid,0.75,0.25,273,0.348251,0.344130,0.186813,0.450549,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: MPNet (273 questions)

In [12]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("mpnet"))
else:
    print("Dev results pending: MPNet")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,mpnet_dense,mpnet,dense,NaN,NaN,273,0.267502,0.263432,0.142857,0.336996,...,273,273,273,273,273,273,273,273,273,273
1,mpnet_hybrid_d0.05_b0.95,mpnet,hybrid,0.05,0.95,273,0.549972,0.547618,0.362637,0.725275,...,273,273,273,273,273,273,273,273,273,273
2,mpnet_hybrid_d0.10_b0.90,mpnet,hybrid,0.10,0.90,273,0.524658,0.522369,0.340659,0.681319,...,273,273,273,273,273,273,273,273,273,273
3,mpnet_hybrid_d0.20_b0.80,mpnet,hybrid,0.20,0.80,273,0.502570,0.500372,0.333333,0.652015,...,273,273,273,273,273,273,273,273,273,273
4,mpnet_hybrid_d0.25_b0.75,mpnet,hybrid,0.25,0.75,273,0.487283,0.485451,0.326007,0.604396,...,273,273,273,273,273,273,273,273,273,273
5,mpnet_hybrid_d0.50_b0.50,mpnet,hybrid,0.50,0.50,273,0.412844,0.411012,0.252747,0.505495,...,273,273,273,273,273,273,273,273,273,273
6,mpnet_hybrid_d0.75_b0.25,mpnet,hybrid,0.75,0.25,273,0.337057,0.333394,0.179487,0.424908,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: E5-base-v2 (273 questions)

In [13]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("e5_base"))
else:
    print("Dev results pending: E5-base-v2")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,e5_base_dense,e5_base,dense,NaN,NaN,273,0.290184,0.287284,0.146520,0.384615,...,273,273,273,273,273,273,273,273,273,273
1,e5_base_hybrid_d0.05_b0.95,e5_base,hybrid,0.05,0.95,273,0.560011,0.557264,0.384615,0.710623,...,273,273,273,273,273,273,273,273,273,273
2,e5_base_hybrid_d0.10_b0.90,e5_base,hybrid,0.10,0.90,273,0.540659,0.537607,0.355311,0.684982,...,273,273,273,273,273,273,273,273,273,273
3,e5_base_hybrid_d0.20_b0.80,e5_base,hybrid,0.20,0.80,273,0.505628,0.502576,0.322344,0.655678,...,273,273,273,273,273,273,273,273,273,273
4,e5_base_hybrid_d0.25_b0.75,e5_base,hybrid,0.25,0.75,273,0.491097,0.488044,0.311355,0.641026,...,273,273,273,273,273,273,273,273,273,273
5,e5_base_hybrid_d0.50_b0.50,e5_base,hybrid,0.50,0.50,273,0.435817,0.432765,0.274725,0.545788,...,273,273,273,273,273,273,273,273,273,273
6,e5_base_hybrid_d0.75_b0.25,e5_base,hybrid,0.75,0.25,273,0.346939,0.342543,0.175824,0.446886,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: BGE-base-en-v1.5 (273 questions)

In [14]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("bge_base"))
else:
    print("Dev results pending: BGE-base-en-v1.5")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_base_dense,bge_base,dense,NaN,NaN,273,0.371125,0.368316,0.216117,0.465201,...,273,273,273,273,273,273,273,273,273,273
1,bge_base_hybrid_d0.05_b0.95,bge_base,hybrid,0.05,0.95,273,0.574955,0.572600,0.406593,0.728938,...,273,273,273,273,273,273,273,273,273,273
2,bge_base_hybrid_d0.10_b0.90,bge_base,hybrid,0.10,0.90,273,0.566106,0.563664,0.391941,0.732601,...,273,273,273,273,273,273,273,273,273,273
3,bge_base_hybrid_d0.20_b0.80,bge_base,hybrid,0.20,0.80,273,0.542834,0.540392,0.366300,0.699634,...,273,273,273,273,273,273,273,273,273,273
4,bge_base_hybrid_d0.25_b0.75,bge_base,hybrid,0.25,0.75,273,0.529310,0.526868,0.355311,0.673993,...,273,273,273,273,273,273,273,273,273,273
5,bge_base_hybrid_d0.50_b0.50,bge_base,hybrid,0.50,0.50,273,0.483214,0.481383,0.318681,0.604396,...,273,273,273,273,273,273,273,273,273,273
6,bge_base_hybrid_d0.75_b0.25,bge_base,hybrid,0.75,0.25,273,0.418471,0.414808,0.238095,0.538462,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: Jina-v2-small (273 questions)

In [15]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("jina"))
else:
    print("Dev results pending: Jina-v2-small")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,jina_dense,jina,dense,NaN,NaN,273,0.312265,0.308759,0.168498,0.402930,...,273,273,273,273,273,273,273,273,273,273
1,jina_hybrid_d0.05_b0.95,jina,hybrid,0.05,0.95,273,0.557881,0.555439,0.377289,0.732601,...,273,273,273,273,273,273,273,273,273,273
2,jina_hybrid_d0.10_b0.90,jina,hybrid,0.10,0.90,273,0.544886,0.542532,0.366300,0.692308,...,273,273,273,273,273,273,273,273,273,273
3,jina_hybrid_d0.20_b0.80,jina,hybrid,0.20,0.80,273,0.530556,0.528201,0.358974,0.677656,...,273,273,273,273,273,273,273,273,273,273
4,jina_hybrid_d0.25_b0.75,jina,hybrid,0.25,0.75,273,0.517757,0.515402,0.347985,0.666667,...,273,273,273,273,273,273,273,273,273,273
5,jina_hybrid_d0.50_b0.50,jina,hybrid,0.50,0.50,273,0.449571,0.447282,0.282051,0.560440,...,273,273,273,273,273,273,273,273,273,273
6,jina_hybrid_d0.75_b0.25,jina,hybrid,0.75,0.25,273,0.393270,0.389607,0.219780,0.505495,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: GTE-ModernBERT (273 questions)

In [16]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("gte_modernbert"))
else:
    print("Dev results pending: GTE-ModernBERT")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,gte_modernbert_dense,gte_modernbert,dense,NaN,NaN,273,0.288416,0.286855,0.146520,0.380952,...,273,273,273,273,273,273,273,273,273,273
1,gte_modernbert_hybrid_d0.05_b0.95,gte_modernbert,hybrid,0.05,0.95,273,0.541283,0.538841,0.347985,0.717949,...,273,273,273,273,273,273,273,273,273,273
2,gte_modernbert_hybrid_d0.10_b0.90,gte_modernbert,hybrid,0.10,0.90,273,0.519805,0.517450,0.336996,0.666667,...,273,273,273,273,273,273,273,273,273,273
3,gte_modernbert_hybrid_d0.20_b0.80,gte_modernbert,hybrid,0.20,0.80,273,0.496321,0.493513,0.322344,0.626374,...,273,273,273,273,273,273,273,273,273,273
4,gte_modernbert_hybrid_d0.25_b0.75,gte_modernbert,hybrid,0.25,0.75,273,0.483028,0.480220,0.315018,0.608059,...,273,273,273,273,273,273,273,273,273,273
5,gte_modernbert_hybrid_d0.50_b0.50,gte_modernbert,hybrid,0.50,0.50,273,0.435451,0.432755,0.282051,0.527473,...,273,273,273,273,273,273,273,273,273,273
6,gte_modernbert_hybrid_d0.75_b0.25,gte_modernbert,hybrid,0.75,0.25,273,0.370783,0.365940,0.212454,0.479853,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: Nomic-v1.5 (273 questions)

In [17]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("nomic"))
else:
    print("Dev results pending: Nomic-v1.5")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,nomic_dense,nomic,dense,NaN,NaN,273,0.303787,0.302356,0.131868,0.417582,...,273,273,273,273,273,273,273,273,273,273
1,nomic_hybrid_d0.05_b0.95,nomic,hybrid,0.05,0.95,273,0.569104,0.566662,0.399267,0.721612,...,273,273,273,273,273,273,273,273,273,273
2,nomic_hybrid_d0.10_b0.90,nomic,hybrid,0.10,0.90,273,0.546036,0.543594,0.369963,0.692308,...,273,273,273,273,273,273,273,273,273,273
3,nomic_hybrid_d0.20_b0.80,nomic,hybrid,0.20,0.80,273,0.523222,0.520867,0.347985,0.663004,...,273,273,273,273,273,273,273,273,273,273
4,nomic_hybrid_d0.25_b0.75,nomic,hybrid,0.25,0.75,273,0.519939,0.517741,0.358974,0.648352,...,273,273,273,273,273,273,273,273,273,273
5,nomic_hybrid_d0.50_b0.50,nomic,hybrid,0.50,0.50,273,0.467164,0.465332,0.311355,0.567766,...,273,273,273,273,273,273,273,273,273,273
6,nomic_hybrid_d0.75_b0.25,nomic,hybrid,0.75,0.25,273,0.398000,0.394337,0.227106,0.509158,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: BGE-M3 (273 questions)

In [18]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("bge_m3"))
else:
    print("Dev results pending: BGE-M3")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_m3_dense,bge_m3,dense,NaN,NaN,273,0.354688,0.351391,0.183150,0.468864,...,273,273,273,273,273,273,273,273,273,273
1,bge_m3_hybrid_d0.05_b0.95,bge_m3,hybrid,0.05,0.95,273,0.576345,0.573990,0.399267,0.747253,...,273,273,273,273,273,273,273,273,273,273
2,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid,0.10,0.90,273,0.570276,0.567921,0.391941,0.736264,...,273,273,273,273,273,273,273,273,273,273
3,bge_m3_hybrid_d0.20_b0.80,bge_m3,hybrid,0.20,0.80,273,0.553724,0.551369,0.373626,0.706960,...,273,273,273,273,273,273,273,273,273,273
4,bge_m3_hybrid_d0.25_b0.75,bge_m3,hybrid,0.25,0.75,273,0.544058,0.541703,0.362637,0.692308,...,273,273,273,273,273,273,273,273,273,273
5,bge_m3_hybrid_d0.50_b0.50,bge_m3,hybrid,0.50,0.50,273,0.471850,0.469652,0.289377,0.615385,...,273,273,273,273,273,273,273,273,273,273
6,bge_m3_hybrid_d0.75_b0.25,bge_m3,hybrid,0.75,0.25,273,0.405749,0.402086,0.216117,0.534799,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev: Qwen3-Embedding-0.6B (273 questions)

In [19]:
if RUN_EXPENSIVE or saved_result_exists("dev_summary.csv"):
    display(model_table("qwen3"))
else:
    print("Dev results pending: Qwen3-Embedding-0.6B")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,qwen3_dense,qwen3,dense,NaN,NaN,273,0.298837,0.298532,0.168498,0.369963,...,273,273,273,273,273,273,273,273,273,273
1,qwen3_hybrid_d0.05_b0.95,qwen3,hybrid,0.05,0.95,273,0.560791,0.558437,0.380952,0.743590,...,273,273,273,273,273,273,273,273,273,273
2,qwen3_hybrid_d0.10_b0.90,qwen3,hybrid,0.10,0.90,273,0.547853,0.545498,0.362637,0.717949,...,273,273,273,273,273,273,273,273,273,273
3,qwen3_hybrid_d0.20_b0.80,qwen3,hybrid,0.20,0.80,273,0.532573,0.530375,0.355311,0.684982,...,273,273,273,273,273,273,273,273,273,273
4,qwen3_hybrid_d0.25_b0.75,qwen3,hybrid,0.25,0.75,273,0.525105,0.523273,0.351648,0.677656,...,273,273,273,273,273,273,273,273,273,273
5,qwen3_hybrid_d0.50_b0.50,qwen3,hybrid,0.50,0.50,273,0.474721,0.472523,0.307692,0.589744,...,273,273,273,273,273,273,273,273,273,273
6,qwen3_hybrid_d0.75_b0.25,qwen3,hybrid,0.75,0.25,273,0.422751,0.421327,0.263736,0.549451,...,273,273,273,273,273,273,273,273,273,273


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Dev overall summary - all 64 configurations

All models and weights are in one table, ordered by **complete evidence@5**, then **MRR**, then run name. This is the same Dev selection order used for the frozen configuration. Every metric column remains available by scrolling horizontally.

In [20]:
def overall_table(split: str) -> pd.DataFrame:
    assert split in ("dev", "proposed_test_all"), split
    live_name = "dev_summary" if split == "dev" else "proposed_test_all_summary"
    if live_name in globals():
        frame = globals()[live_name]
    else:
        saved = result_path(f"{split}_summary.csv")
        if not saved.is_file():
            raise FileNotFoundError(f"Run the full comparison first: {saved}")
        frame = pd.read_csv(saved)
    table = frame.copy()
    table = table.sort_values(["complete_hit_at_5", "mrr", "run_name"],
                              ascending=[False, False, True]).reset_index(drop=True)
    table["mode"] = table["mode"].map(lambda mode: "hybrid" if mode.startswith("hybrid_") else mode)
    leading = ["run_name", "model", "mode", "dense_weight", "bm25_weight", "question_count"]
    columns = leading + [column for column in table.columns if column not in leading]
    return table[columns]

if "dev_summary" in globals() or saved_result_exists("dev_summary.csv"):
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(overall_table("dev"))
else:
    print("Dev overall summary pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,hit_at_5,hit_at_10,complete_evidence_hit_at_1,complete_evidence_hit_at_3,complete_evidence_hit_at_5,complete_evidence_hit_at_10,evidence_recall_at_1,evidence_recall_at_3,evidence_recall_at_5,evidence_recall_at_10,first_hit_mrr,complete_hit_at_1,complete_hit_at_3,complete_hit_at_5,complete_hit_at_10,recall_at_1,recall_at_3,recall_at_5,recall_at_10,core_precision_at_1,core_precision_at_3,core_precision_at_5,core_precision_at_10,context_noise_rate_at_1,context_noise_rate_at_3,context_noise_rate_at_5,context_noise_rate_at_10,no_relevant_hit_rate_at_1,no_relevant_hit_rate_at_3,no_relevant_hit_rate_at_5,no_relevant_hit_rate_at_10,partial_evidence_count_at_1,partial_evidence_count_at_3,partial_evidence_count_at_5,partial_evidence_count_at_10,returned_count_at_1,returned_count_at_3,returned_count_at_5,returned_count_at_10,empty_retrieval_rate_at_1,empty_retrieval_rate_at_3,empty_retrieval_rate_at_5,empty_retrieval_rate_at_10,no_hit_rate_at_1,no_hit_rate_at_3,no_hit_rate_at_5,no_hit_rate_at_10,returned_count,empty_retrieval,unanswerable_no_hit,unanswerable_false_positive,no_relevant_hit_at_1,no_relevant_hit_at_3,no_relevant_hit_at_5,no_relevant_hit_at_10,core_precision_at_1_defined_count,core_precision_at_3_defined_count,core_precision_at_5_defined_count,core_precision_at_10_defined_count,context_noise_rate_at_1_defined_count,context_noise_rate_at_3_defined_count,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid,0.10,0.90,273,0.570276,0.567921,0.391941,0.736264,0.794872,0.853480,0.391941,0.732601,0.791209,0.846154,0.391941,0.734432,0.793040,0.849817,0.570276,0.391941,0.732601,0.791209,0.846154,0.391941,0.734432,0.793040,0.849817,0.391941,0.245421,0.158974,0.085348,0.608059,0.754579,0.841026,0.914652,0.608059,0.263736,0.205128,0.146520,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.608059,0.263736,0.205128,0.146520,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273
1,qwen3_hybrid_d0.05_b0.95,qwen3,hybrid,0.05,0.95,273,0.560791,0.558437,0.380952,0.743590,0.794872,0.849817,0.380952,0.739927,0.791209,0.842491,0.380952,0.741758,0.793040,0.846154,0.560791,0.380952,0.739927,0.791209,0.842491,0.380952,0.741758,0.793040,0.846154,0.380952,0.247863,0.158974,0.084982,0.619048,0.752137,0.841026,0.915018,0.619048,0.256410,0.205128,0.150183,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.619048,0.256410,0.205128,0.150183,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273
2,gte_modernbert_hybrid_d0.05_b0.95,gte_modernbert,hybrid,0.05,0.95,273,0.541283,0.538841,0.347985,0.717949,0.794872,0.849817,0.347985,0.714286,0.791209,0.842491,0.347985,0.716117,0.793040,0.846154,0.541283,0.347985,0.714286,0.791209,0.842491,0.347985,0.716117,0.793040,0.846154,0.347985,0.239316,0.158974,0.084982,0.652015,0.760684,0.841026,0.915018,0.652015,0.282051,0.205128,0.150183,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.652015,0.282051,0.205128,0.150183,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273,273
3,bge_m3_hybrid_d0.05_b0.95,bge_m3,hybrid,0.05,0.95,273,0.576345,0.573990,0.399267,0.747253,0.791209,0.849817,0.399267,0.743590,0.787546,0.842491,0.399267,0.745421,0.789377,0.846154,0.576345,0.399267,0.743590,0.787546,0.842491,0.399267,0.745421,0.789377,0.846154,0.399267,0.249084,0.158242,0.084982,0.600733,0.750916,0.841758,0.915018,0.600733,0.252747,0.208791,0.150183,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10

## 9. Freeze a diagnostic configuration, then evaluate stratified Test

First freeze one configuration using only the 273-question Dev split. The
68-question Test split is evaluated only after that choice. The later all-model
Test comparison is exploratory and must not be used to retune the frozen choice.
This remains diagnostic because the corpus, Gold annotations, and split are not
formally approved releases.


In [21]:
if RUN_EXPENSIVE:
    # BM25 is a legitimate configuration: do not exclude a stronger baseline.
    winner = dev_summary.iloc[0].to_dict()
    frozen = {"model": winner["model"], "mode": winner["mode"],
              "bm25_weight": None if pd.isna(winner["bm25_weight"]) else float(winner["bm25_weight"]),
              "dense_weight": None if pd.isna(winner["dense_weight"]) else float(winner["dense_weight"]),
              "selected_on": "273-question stratified diagnostic development only",
              "selection_metric": "complete_hit_at_5 then mrr"}
    print("Frozen diagnostic configuration:", frozen)
    holdout_bm25 = {q["question_id"]: bm25_search(q["question"], SEARCH_DEPTH) for q in heldout_gold}
    if frozen["model"] != "bm25":
        model, index, _ = load_or_build_dense(frozen["model"])
    holdout_rows, holdout_candidates = [], []
    for q in heldout_gold:
        if frozen["model"] == "bm25":
            hits = holdout_bm25[q["question_id"]][:RETRIEVAL_DEPTH]
        else:
            dense = dense_search(q["question"], frozen["model"], model, index)
            hits = dense[:RETRIEVAL_DEPTH] if frozen["mode"] == "dense" else weighted_rrf(
                holdout_bm25[q["question_id"]], dense, frozen["bm25_weight"], frozen["dense_weight"])
        holdout_rows.append(score_one(q, hits, frozen["model"], frozen["mode"],
                                      frozen["bm25_weight"], frozen["dense_weight"]))
        for rank, (chunk_id, score) in enumerate(hits, 1):
            holdout_candidates.append({"question_id": q["question_id"], "model": frozen["model"],
                "mode": frozen["mode"], "rank": rank, "chunk_id": chunk_id, "score": score,
                "query": q["question"], "corpus_hash": corpus_manifest["corpus_hash"],
                "textbook_id": by_chunk[chunk_id]["textbook_id"], "provider": by_chunk[chunk_id]["provider"],
                "source_locator": by_chunk[chunk_id]["source_locator"]})
    if frozen["model"] != "bm25":
        del model, index
    display(summarize(holdout_rows))
else:
    holdout_rows = holdout_candidates = []
    manifest_path = result_path("run_manifest.json")
    assert manifest_path.is_file(), "Committed run manifest is missing."
    saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    frozen = saved_manifest["frozen_diagnostic_configuration"]
    print("Frozen diagnostic configuration from committed manifest:", frozen)
    display(pd.read_csv(result_path("proposed_holdout_summary.csv")))

Frozen diagnostic configuration from committed manifest: {'model': 'bge_m3', 'mode': 'hybrid_d0.10_b0.90', 'bm25_weight': 0.9, 'dense_weight': 0.1, 'selected_on': '273-question stratified diagnostic development only', 'selection_metric': 'complete_hit_at_5 then mrr'}


,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid_d0.10_b0.90,0.1,0.9,68,0.471499,0.46966,0.279412,0.647059,...,68,68,68,68,68,68,68,68,68,68


## 10. Stratified Test (68 questions): all-model exploratory comparison

This section runs BM25 and **all nine embedding models** on the 68-question
group-stratified Test split. Each model has pure Dense plus the same six
Dense/BM25 hybrids used for Dev. Test contains all six question types and uses
the textbook quotas 25/19/17/7. No rebound Golden Evidence chunk appears in
both Dev and Test. The all-model Test table is exploratory; the single
configuration frozen before this comparison remains the unbiased diagnostic
holdout result in Section 9.


In [22]:
if RUN_EXPENSIVE:
    proposed_test_all_rows, proposed_test_all_candidates, test_index_provenance = evaluate_questions(
        heldout_gold, list(MODEL_SPECS), checkpoint_prefix="proposed_test_all")
    assert set(test_index_provenance) == set(index_provenance)
    proposed_test_all_summary = summarize(proposed_test_all_rows)
    print("Exploratory proposed-Test configurations:", len(proposed_test_all_summary))
    print("Questions per configuration:", sorted(proposed_test_all_summary["question_count"].unique()))
else:
    proposed_test_all_rows = proposed_test_all_candidates = []
    print("No new proposed-Test comparison run; saved tables below are shown if available.")

No new proposed-Test comparison run; saved tables below are shown if available.


### Stratified Test: BM25 baseline (68 questions; exploratory)

In [23]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("bm25", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bm25_bm25,bm25,bm25,NaN,NaN,68,0.500654,0.500654,0.308824,0.691176,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: MiniLM (68 questions; exploratory)

In [24]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("minilm", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,minilm_dense,minilm,dense,NaN,NaN,68,0.244118,0.239216,0.102941,0.338235,...,68,68,68,68,68,68,68,68,68,68
1,minilm_hybrid_d0.05_b0.95,minilm,hybrid,0.05,0.95,68,0.478145,0.478145,0.294118,0.617647,...,68,68,68,68,68,68,68,68,68,68
2,minilm_hybrid_d0.10_b0.90,minilm,hybrid,0.10,0.90,68,0.434547,0.433077,0.264706,0.500000,...,68,68,68,68,68,68,68,68,68,68
3,minilm_hybrid_d0.20_b0.80,minilm,hybrid,0.20,0.80,68,0.413229,0.411595,0.264706,0.500000,...,68,68,68,68,68,68,68,68,68,68
4,minilm_hybrid_d0.25_b0.75,minilm,hybrid,0.25,0.75,68,0.387086,0.385452,0.250000,0.485294,...,68,68,68,68,68,68,68,68,68,68
5,minilm_hybrid_d0.50_b0.50,minilm,hybrid,0.50,0.50,68,0.390313,0.375607,0.264706,0.500000,...,68,68,68,68,68,68,68,68,68,68
6,minilm_hybrid_d0.75_b0.25,minilm,hybrid,0.75,0.25,68,0.311193,0.296487,0.161765,0.441176,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: MPNet (68 questions; exploratory)

In [25]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("mpnet", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,mpnet_dense,mpnet,dense,NaN,NaN,68,0.222771,0.219094,0.073529,0.352941,...,68,68,68,68,68,68,68,68,68,68
1,mpnet_hybrid_d0.05_b0.95,mpnet,hybrid,0.05,0.95,68,0.473080,0.473080,0.294118,0.647059,...,68,68,68,68,68,68,68,68,68,68
2,mpnet_hybrid_d0.10_b0.90,mpnet,hybrid,0.10,0.90,68,0.454470,0.453000,0.294118,0.558824,...,68,68,68,68,68,68,68,68,68,68
3,mpnet_hybrid_d0.20_b0.80,mpnet,hybrid,0.20,0.80,68,0.429610,0.427159,0.279412,0.529412,...,68,68,68,68,68,68,68,68,68,68
4,mpnet_hybrid_d0.25_b0.75,mpnet,hybrid,0.25,0.75,68,0.420606,0.418155,0.279412,0.514706,...,68,68,68,68,68,68,68,68,68,68
5,mpnet_hybrid_d0.50_b0.50,mpnet,hybrid,0.50,0.50,68,0.401815,0.394462,0.294118,0.455882,...,68,68,68,68,68,68,68,68,68,68
6,mpnet_hybrid_d0.75_b0.25,mpnet,hybrid,0.75,0.25,68,0.291970,0.284617,0.132353,0.411765,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: E5-base-v2 (68 questions; exploratory)

In [26]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("e5_base", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,e5_base_dense,e5_base,dense,NaN,NaN,68,0.262763,0.262763,0.147059,0.338235,...,68,68,68,68,68,68,68,68,68,68
1,e5_base_hybrid_d0.05_b0.95,e5_base,hybrid,0.05,0.95,68,0.486356,0.486356,0.294118,0.676471,...,68,68,68,68,68,68,68,68,68,68
2,e5_base_hybrid_d0.10_b0.90,e5_base,hybrid,0.10,0.90,68,0.482353,0.480719,0.294118,0.647059,...,68,68,68,68,68,68,68,68,68,68
3,e5_base_hybrid_d0.20_b0.80,e5_base,hybrid,0.20,0.80,68,0.457592,0.455958,0.279412,0.602941,...,68,68,68,68,68,68,68,68,68,68
4,e5_base_hybrid_d0.25_b0.75,e5_base,hybrid,0.25,0.75,68,0.434028,0.432557,0.250000,0.602941,...,68,68,68,68,68,68,68,68,68,68
5,e5_base_hybrid_d0.50_b0.50,e5_base,hybrid,0.50,0.50,68,0.386537,0.386537,0.220588,0.529412,...,68,68,68,68,68,68,68,68,68,68
6,e5_base_hybrid_d0.75_b0.25,e5_base,hybrid,0.75,0.25,68,0.338124,0.336654,0.191176,0.426471,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: BGE-base-en-v1.5 (68 questions; exploratory)

In [27]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("bge_base", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_base_dense,bge_base,dense,NaN,NaN,68,0.305363,0.305363,0.176471,0.382353,...,68,68,68,68,68,68,68,68,68,68
1,bge_base_hybrid_d0.05_b0.95,bge_base,hybrid,0.05,0.95,68,0.501797,0.501797,0.308824,0.691176,...,68,68,68,68,68,68,68,68,68,68
2,bge_base_hybrid_d0.10_b0.90,bge_base,hybrid,0.10,0.90,68,0.478694,0.478694,0.279412,0.691176,...,68,68,68,68,68,68,68,68,68,68
3,bge_base_hybrid_d0.20_b0.80,bge_base,hybrid,0.20,0.80,68,0.457475,0.455637,0.279412,0.602941,...,68,68,68,68,68,68,68,68,68,68
4,bge_base_hybrid_d0.25_b0.75,bge_base,hybrid,0.25,0.75,68,0.452982,0.451348,0.279412,0.617647,...,68,68,68,68,68,68,68,68,68,68
5,bge_base_hybrid_d0.50_b0.50,bge_base,hybrid,0.50,0.50,68,0.406332,0.404493,0.250000,0.544118,...,68,68,68,68,68,68,68,68,68,68
6,bge_base_hybrid_d0.75_b0.25,bge_base,hybrid,0.75,0.25,68,0.343091,0.343091,0.176471,0.485294,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: Jina-v2-small (68 questions; exploratory)

In [28]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("jina", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,jina_dense,jina,dense,NaN,NaN,68,0.239595,0.239595,0.147059,0.279412,...,68,68,68,68,68,68,68,68,68,68
1,jina_hybrid_d0.05_b0.95,jina,hybrid,0.05,0.95,68,0.476389,0.474918,0.279412,0.676471,...,68,68,68,68,68,68,68,68,68,68
2,jina_hybrid_d0.10_b0.90,jina,hybrid,0.10,0.90,68,0.466258,0.464624,0.279412,0.617647,...,68,68,68,68,68,68,68,68,68,68
3,jina_hybrid_d0.20_b0.80,jina,hybrid,0.20,0.80,68,0.416480,0.414029,0.220588,0.544118,...,68,68,68,68,68,68,68,68,68,68
4,jina_hybrid_d0.25_b0.75,jina,hybrid,0.25,0.75,68,0.387803,0.385352,0.205882,0.529412,...,68,68,68,68,68,68,68,68,68,68
5,jina_hybrid_d0.50_b0.50,jina,hybrid,0.50,0.50,68,0.347602,0.340249,0.191176,0.455882,...,68,68,68,68,68,68,68,68,68,68
6,jina_hybrid_d0.75_b0.25,jina,hybrid,0.75,0.25,68,0.308917,0.304015,0.176471,0.382353,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: GTE-ModernBERT (68 questions; exploratory)

In [29]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("gte_modernbert", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,gte_modernbert_dense,gte_modernbert,dense,NaN,NaN,68,0.257750,0.256279,0.147059,0.308824,...,68,68,68,68,68,68,68,68,68,68
1,gte_modernbert_hybrid_d0.05_b0.95,gte_modernbert,hybrid,0.05,0.95,68,0.471446,0.471446,0.264706,0.661765,...,68,68,68,68,68,68,68,68,68,68
2,gte_modernbert_hybrid_d0.10_b0.90,gte_modernbert,hybrid,0.10,0.90,68,0.449953,0.448319,0.250000,0.602941,...,68,68,68,68,68,68,68,68,68,68
3,gte_modernbert_hybrid_d0.20_b0.80,gte_modernbert,hybrid,0.20,0.80,68,0.428922,0.426471,0.250000,0.544118,...,68,68,68,68,68,68,68,68,68,68
4,gte_modernbert_hybrid_d0.25_b0.75,gte_modernbert,hybrid,0.25,0.75,68,0.409250,0.406799,0.235294,0.514706,...,68,68,68,68,68,68,68,68,68,68
5,gte_modernbert_hybrid_d0.50_b0.50,gte_modernbert,hybrid,0.50,0.50,68,0.366970,0.363294,0.191176,0.500000,...,68,68,68,68,68,68,68,68,68,68
6,gte_modernbert_hybrid_d0.75_b0.25,gte_modernbert,hybrid,0.75,0.25,68,0.326809,0.323868,0.191176,0.367647,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: Nomic-v1.5 (68 questions; exploratory)

In [30]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("nomic", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,nomic_dense,nomic,dense,NaN,NaN,68,0.265575,0.265575,0.161765,0.323529,...,68,68,68,68,68,68,68,68,68,68
1,nomic_hybrid_d0.05_b0.95,nomic,hybrid,0.05,0.95,68,0.484903,0.484903,0.294118,0.647059,...,68,68,68,68,68,68,68,68,68,68
2,nomic_hybrid_d0.10_b0.90,nomic,hybrid,0.10,0.90,68,0.468622,0.468622,0.279412,0.632353,...,68,68,68,68,68,68,68,68,68,68
3,nomic_hybrid_d0.20_b0.80,nomic,hybrid,0.20,0.80,68,0.414560,0.412722,0.220588,0.588235,...,68,68,68,68,68,68,68,68,68,68
4,nomic_hybrid_d0.25_b0.75,nomic,hybrid,0.25,0.75,68,0.410819,0.408368,0.220588,0.573529,...,68,68,68,68,68,68,68,68,68,68
5,nomic_hybrid_d0.50_b0.50,nomic,hybrid,0.50,0.50,68,0.385697,0.382020,0.250000,0.485294,...,68,68,68,68,68,68,68,68,68,68
6,nomic_hybrid_d0.75_b0.25,nomic,hybrid,0.75,0.25,68,0.336006,0.331104,0.191176,0.441176,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: BGE-M3 (68 questions; exploratory)

In [31]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("bge_m3", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,bge_m3_dense,bge_m3,dense,NaN,NaN,68,0.328799,0.325123,0.191176,0.411765,...,68,68,68,68,68,68,68,68,68,68
1,bge_m3_hybrid_d0.05_b0.95,bge_m3,hybrid,0.05,0.95,68,0.495279,0.495279,0.308824,0.661765,...,68,68,68,68,68,68,68,68,68,68
2,bge_m3_hybrid_d0.10_b0.90,bge_m3,hybrid,0.10,0.90,68,0.471499,0.469660,0.279412,0.647059,...,68,68,68,68,68,68,68,68,68,68
3,bge_m3_hybrid_d0.20_b0.80,bge_m3,hybrid,0.20,0.80,68,0.453186,0.448284,0.264706,0.647059,...,68,68,68,68,68,68,68,68,68,68
4,bge_m3_hybrid_d0.25_b0.75,bge_m3,hybrid,0.25,0.75,68,0.447035,0.442134,0.264706,0.632353,...,68,68,68,68,68,68,68,68,68,68
5,bge_m3_hybrid_d0.50_b0.50,bge_m3,hybrid,0.50,0.50,68,0.411817,0.404464,0.250000,0.529412,...,68,68,68,68,68,68,68,68,68,68
6,bge_m3_hybrid_d0.75_b0.25,bge_m3,hybrid,0.75,0.25,68,0.375053,0.367700,0.220588,0.470588,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test: Qwen3-Embedding-0.6B (68 questions; exploratory)

In [32]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    display(model_table("qwen3", split="proposed_test_all"))
else:
    print("Proposed-Test comparison pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,...,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,qwen3_dense,qwen3,dense,NaN,NaN,68,0.234979,0.234979,0.147059,0.250000,...,68,68,68,68,68,68,68,68,68,68
1,qwen3_hybrid_d0.05_b0.95,qwen3,hybrid,0.05,0.95,68,0.501879,0.501879,0.323529,0.661765,...,68,68,68,68,68,68,68,68,68,68
2,qwen3_hybrid_d0.10_b0.90,qwen3,hybrid,0.10,0.90,68,0.481513,0.481513,0.308824,0.617647,...,68,68,68,68,68,68,68,68,68,68
3,qwen3_hybrid_d0.20_b0.80,qwen3,hybrid,0.20,0.80,68,0.448979,0.447508,0.279412,0.573529,...,68,68,68,68,68,68,68,68,68,68
4,qwen3_hybrid_d0.25_b0.75,qwen3,hybrid,0.25,0.75,68,0.426021,0.424551,0.250000,0.573529,...,68,68,68,68,68,68,68,68,68,68
5,qwen3_hybrid_d0.50_b0.50,qwen3,hybrid,0.50,0.50,68,0.376418,0.376418,0.220588,0.455882,...,68,68,68,68,68,68,68,68,68,68
6,qwen3_hybrid_d0.75_b0.25,qwen3,hybrid,0.75,0.25,68,0.307861,0.307861,0.161765,0.367647,...,68,68,68,68,68,68,68,68,68,68


The executed table above contains the measured four-book results. Machine-readable summaries are saved under `docs/results/m6_w6_v2_four_book_gold_v0_1_3/`.

### Stratified Test overall summary - all 64 configurations

All nine models, six hybrids per model, pure Dense and BM25 are in one table, ordered by **complete evidence@5**, then **MRR**, then run name. This order is for inspection only: the proposed Test results were **not** used to change the Dev-frozen configuration. Every metric column remains available by scrolling horizontally.

In [33]:
if "proposed_test_all_summary" in globals() or saved_result_exists("proposed_test_all_summary.csv"):
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(overall_table("proposed_test_all"))
else:
    print("Proposed-Test overall summary pending. Run the full notebook first.")

,run_name,model,mode,dense_weight,bm25_weight,question_count,mrr,complete_evidence_mrr,hit_at_1,hit_at_3,hit_at_5,hit_at_10,complete_evidence_hit_at_1,complete_evidence_hit_at_3,complete_evidence_hit_at_5,complete_evidence_hit_at_10,evidence_recall_at_1,evidence_recall_at_3,evidence_recall_at_5,evidence_recall_at_10,first_hit_mrr,complete_hit_at_1,complete_hit_at_3,complete_hit_at_5,complete_hit_at_10,recall_at_1,recall_at_3,recall_at_5,recall_at_10,core_precision_at_1,core_precision_at_3,core_precision_at_5,core_precision_at_10,context_noise_rate_at_1,context_noise_rate_at_3,context_noise_rate_at_5,context_noise_rate_at_10,no_relevant_hit_rate_at_1,no_relevant_hit_rate_at_3,no_relevant_hit_rate_at_5,no_relevant_hit_rate_at_10,partial_evidence_count_at_1,partial_evidence_count_at_3,partial_evidence_count_at_5,partial_evidence_count_at_10,returned_count_at_1,returned_count_at_3,returned_count_at_5,returned_count_at_10,empty_retrieval_rate_at_1,empty_retrieval_rate_at_3,empty_retrieval_rate_at_5,empty_retrieval_rate_at_10,no_hit_rate_at_1,no_hit_rate_at_3,no_hit_rate_at_5,no_hit_rate_at_10,returned_count,empty_retrieval,unanswerable_no_hit,unanswerable_false_positive,no_relevant_hit_at_1,no_relevant_hit_at_3,no_relevant_hit_at_5,no_relevant_hit_at_10,core_precision_at_1_defined_count,core_precision_at_3_defined_count,core_precision_at_5_defined_count,core_precision_at_10_defined_count,context_noise_rate_at_1_defined_count,context_noise_rate_at_3_defined_count,context_noise_rate_at_5_defined_count,context_noise_rate_at_10_defined_count,no_relevant_hit_rate_at_1_defined_count,no_relevant_hit_rate_at_3_defined_count,no_relevant_hit_rate_at_5_defined_count,no_relevant_hit_rate_at_10_defined_count,partial_evidence_count_at_1_defined_count,partial_evidence_count_at_3_defined_count,partial_evidence_count_at_5_defined_count,partial_evidence_count_at_10_defined_count
0,e5_base_hybrid_d0.05_b0.95,e5_base,hybrid,0.05,0.95,68,0.486356,0.486356,0.294118,0.676471,0.764706,0.794118,0.294118,0.676471,0.764706,0.794118,0.294118,0.676471,0.764706,0.794118,0.486356,0.294118,0.676471,0.764706,0.794118,0.294118,0.676471,0.764706,0.794118,0.294118,0.225490,0.152941,0.079412,0.705882,0.774510,0.847059,0.920588,0.705882,0.323529,0.235294,0.205882,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.705882,0.323529,0.235294,0.205882,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68
1,qwen3_hybrid_d0.05_b0.95,qwen3,hybrid,0.05,0.95,68,0.501879,0.501879,0.323529,0.661765,0.750000,0.794118,0.323529,0.661765,0.750000,0.794118,0.323529,0.661765,0.750000,0.794118,0.501879,0.323529,0.661765,0.750000,0.794118,0.323529,0.661765,0.750000,0.794118,0.323529,0.220588,0.150000,0.079412,0.676471,0.779412,0.850000,0.920588,0.676471,0.338235,0.250000,0.205882,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.676471,0.338235,0.250000,0.205882,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68
2,bge_base_hybrid_d0.05_b0.95,bge_base,hybrid,0.05,0.95,68,0.501797,0.501797,0.308824,0.691176,0.750000,0.794118,0.308824,0.691176,0.750000,0.794118,0.308824,0.691176,0.750000,0.794118,0.501797,0.308824,0.691176,0.750000,0.794118,0.308824,0.691176,0.750000,0.794118,0.308824,0.230392,0.150000,0.079412,0.691176,0.769608,0.850000,0.920588,0.691176,0.308824,0.250000,0.205882,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.691176,0.308824,0.250000,0.205882,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68,68
3,bm25_bm25,bm25,bm25,NaN,NaN,68,0.500654,0.500654,0.308824,0.691176,0.750000,0.794118,0.308824,0.691176,0.750000,0.794118,0.308824,0.691176,0.750000,0.794118,0.500654,0.308824,0.691176,0.750000,0.794118,0.308824,0.691176,0.750000,0.794118,0.308824,0.230392,0.150000,0.079412,0.691176,0.769608,0.850000,0.920588,0.691176,0.308824,0.250000,0.205882,0.0,0.0,0.0,0.0,1.0,3.0,5.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,NaN,NaN,0.691176,0.308824,0.250000,0.205882,68,68,68,68,68,68,68,68,68,68,68,

## 11. Save auditable diagnostic outputs and manifest

In [34]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.write_text("".join(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n" for row in rows), encoding="utf-8")

if RUN_EXPENSIVE:
    assert len(dev_rows) == len(dev_gold) * (1 + len(MODEL_SPECS) * (1 + len(HYBRID_WEIGHTS)))
    assert len(proposed_test_all_rows) == len(heldout_gold) * (1 + len(MODEL_SPECS) * (1 + len(HYBRID_WEIGHTS)))
    assert {r["question_id"] for r in dev_rows} == {q["question_id"] for q in dev_gold}
    assert {r["question_id"] for r in holdout_rows} == {q["question_id"] for q in heldout_gold}
    assert {r["question_id"] for r in proposed_test_all_rows} == {q["question_id"] for q in heldout_gold}
    assert all(r["evaluation_split"] == "stratified_test" for r in proposed_test_all_rows)
    frozen_test_rows = {r["question_id"]: r for r in proposed_test_all_rows
                        if r["model"] == frozen["model"] and r["mode"] == frozen["mode"]}
    assert len(frozen_test_rows) == len(heldout_gold)
    assert all(r["ranked_chunk_ids"] == frozen_test_rows[r["question_id"]]["ranked_chunk_ids"]
               for r in holdout_rows), "Frozen result must agree with the all-model audit."
    split_assignments = [
        {"question_id": q["question_id"], "evaluation_split": q["evaluation_split"],
         "source_gold_split": q["source_gold_split"],
         "textbook_id": question_textbook_id(q),
         "question_type": q["question_type"], "question_difficulty": q["question_difficulty"]}
        for q in dev_gold + heldout_gold
    ]
    write_jsonl(OUTPUT_DIR / "evaluation_split_assignments.jsonl", split_assignments)
    write_jsonl(OUTPUT_DIR / "gold_span_bindings.jsonl", binding_rows)
    write_jsonl(OUTPUT_DIR / "dev_metrics.jsonl", dev_rows)
    write_jsonl(OUTPUT_DIR / "dev_candidate_hits.jsonl", dev_candidates)
    write_jsonl(OUTPUT_DIR / "proposed_holdout_metrics.jsonl", holdout_rows)
    write_jsonl(OUTPUT_DIR / "proposed_holdout_candidate_hits.jsonl", holdout_candidates)
    write_jsonl(OUTPUT_DIR / "proposed_test_all_metrics.jsonl", proposed_test_all_rows)
    write_jsonl(OUTPUT_DIR / "proposed_test_all_candidate_hits.jsonl", proposed_test_all_candidates)
    summarize(dev_rows).to_csv(OUTPUT_DIR / "dev_summary.csv", index=False)
    summarize(dev_rows).sort_values(["complete_hit_at_5", "mrr", "run_name"],
        ascending=[False, False, True]).to_csv(OUTPUT_DIR / "dev_overall_ranked.csv", index=False)
    grouped_summary(dev_rows).to_csv(OUTPUT_DIR / "dev_grouped_summary.csv", index=False)
    summarize(holdout_rows).to_csv(OUTPUT_DIR / "proposed_holdout_summary.csv", index=False)
    proposed_test_all_summary.to_csv(OUTPUT_DIR / "proposed_test_all_summary.csv", index=False)
    proposed_test_all_summary.sort_values(["complete_hit_at_5", "mrr", "run_name"],
        ascending=[False, False, True]).to_csv(OUTPUT_DIR / "proposed_test_all_overall_ranked.csv", index=False)
    run_manifest = {
        "schema_version": "m6-v2-diagnostic-gold-v0.1.3-split-273-68-1", "reportable": False,
        "method_config": METHOD_CONFIG,
        "evaluation_set_version": EVALUATION_SET_VERSION,
        "gold_acceptance_policy": GOLD_ACCEPTANCE_POLICY,
        "source_annotation_status_counts": dict(collections.Counter(g["annotation_status"] for g in gold)),
        "gold_annotation_version": GOLD_ANNOTATION_VERSION,
        "gold_row_count": len(gold),
        "gold_evidence_span_count": len(binding_rows),
        "gold_sha256": sha256(gold_bytes), "github_gold_sha256": sha256(github_gold_bytes),
        "gold_source_corpus_versions": sorted(gold_source_corpus_versions),
        "gold_rebound_to_corpus_version": corpus_manifest["corpus_version"],
        "model_registry": {key: spec["model"] for key, spec in MODEL_SPECS.items()},
        "binding_sha256": sha256(binding_bytes), "github_commit": repo_commit,
        "github_worktree_dirty": repo_worktree_dirty,
        "corpus_version": corpus_manifest["corpus_version"],
        "corpus_hash": corpus_manifest["corpus_hash"],
        "corpus_manifest_hash": corpus_manifest["manifest_hash"],
        "chunk_config_hash": corpus_manifest["chunk_config_hash"],
        "required_textbook_ids": corpus_manifest["required_textbook_ids"],
        "included_textbook_ids": corpus_manifest["included_textbook_ids"],
        "corpus_zip_sha256": sha256(CORPUS_ZIP.read_bytes()),
        "m5_zip_sha256": sha256(M5_ZIP.read_bytes()),
        "indexes": {k: {"embedding_model": a.get("embedding_model", a.get("model")),
                         "embedding_revision": a.get("embedding_revision"),
                         "index_sha256": a.get("metadata", {}).get("index_sha256", a.get("index_sha256")),
                         "normalise_embeddings": a.get("normalise_embeddings"),
                         "similarity_metric": a.get("similarity_metric", a.get("metric"))}
                    for k, a in index_provenance.items()},
        "frozen_diagnostic_configuration": frozen,
        "dev_question_count": len(dev_gold), "test_question_count": len(heldout_gold),
        "proposed_holdout_question_count": len(heldout_gold),
        "split_method": "deterministic textbook-quota group stratification",
        "split_seed": SPLIT_SEED,
        "split_stratify_fields": list(STRATIFY_FIELDS),
        "test_textbook_quotas": TEST_QUOTAS,
        "test_question_ids_sha256": TEST_QUESTION_IDS_SHA256,
        "cross_split_evidence_chunk_count": len(dev_evidence & test_evidence),
        "proposed_test_all_configuration_count": len(proposed_test_all_summary),
        "proposed_test_all_is_exploratory": True,
        "unanswerable_question_count": 0,
        "unsupported_metrics": ["unanswerable_no_hit_rate", "unanswerable_false_positive_rate",
                                "cross_textbook_rate", "cross_provider_rate"],
        "limitations": [
            "derived 273/68 group-stratified split is reproducible but not formally frozen",
            "development corpus is reportable=false",
            "Gold source versions are preserved and all evidence spans are content-verified against this four-book corpus",
            "all questions are answerable and each evidence path is single-textbook/single-provider",
        ],
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    (OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", OUTPUT_DIR)
else:
    print("Preflight only: no new measured files were written; committed results remain unchanged.")

Preflight only: no new measured files were written; committed results remain unchanged.


## Interpretation guardrails

**For actual model scores, use the per-model development tables above.** The
notes below describe interpretation limits, not additional scores.

- This notebook compares retrieval only; it does not evaluate M7/M8 answer
  quality or refusal behavior.
- All 341 Gold v0.1.3 questions are answerable. `unanswerable_no_hit_rate` and
  `unanswerable_false_positive_rate` are **not applicable**, not zero. Empty
  retrieval and false no-relevant-hit on answerable questions are measured
  separately.
- Golden Evidence v0.1.3 contains 341 single-evidence-path questions and 352
  evidence spans: 259 questions from the OpenStax source corpus and 82 CK-12
  questions. It therefore supports textbook-specific CK-12 grouping, but no
  question has a cross-textbook or cross-provider evidence path, so those rates
  cannot be inferred.
- `dev_candidate_hits.jsonl` and
  `proposed_holdout_candidate_hits.jsonl` are real ranked candidate handoffs
  with chunk ID, source and score, generated only after model execution.
- Gold v0.1.3 is accepted as working Gold for this M6 experiment. Its source
  corpus versions and draft annotation status are retained, and all 352 spans
  must pass explicit block/chapter/verbatim rebinding before scoring.
- The derived 273/68 split has zero exact-evidence overlap and preserves textbook quotas, but it is not an officially approved split. For a formal run, obtain an official frozen Dev/Test split, a reportable
  corpus, Gold native to that exact corpus, and approved evaluation-set/Gold
  versions, then enable and satisfy `FORMAL_MODE`.
